# Integration Assignment — Full EDA Pipeline on the Orders Dataset
### Week 5 · Thursday · Review

**Goal:** Run a complete, independent EDA pipeline — diagnose, clean, visualize, summarize — on a larger, messier dataset finding every planted problem myself before fixing anything.

## Step 1: Generate the dataset

The dataset below is built from a fixed, required spec (seeded random generation) so it has known, reproducible mess.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)
n = 5000

orders = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "order_date": pd.date_range("2024-01-01", periods=n, freq="h"),
    "customer_id": rng.integers(1000, 1200, size=n),
    "product_category": rng.choice(
        ["Electronics", "electronics", "Home Goods", "Apparel", "Books"], size=n
    ),
    "quantity": rng.integers(1, 8, size=n),
    "unit_price": rng.normal(45, 20, size=n).round(2),
    "region": rng.choice(["North", "South", "East", "West", None], size=n, p=[0.24, 0.24, 0.24, 0.24, 0.04]),
})

# Introduce the mess, on purpose — do not skip this part
orders.loc[rng.choice(n, 150, replace=False), "customer_id"] = None
orders.loc[rng.choice(n, 30, replace=False), "quantity"] *= -1          # returns, disguised as negative quantity
orders.loc[rng.choice(n, 20, replace=False), "unit_price"] = 4999.99    # data-entry outliers
orders = pd.concat([orders, orders.sample(15, random_state=1)])        # duplicate rows, unannounced

In [2]:
orders.head()

,order_id,order_date,customer_id,product_category,quantity,unit_price,region
0,1,2024-01-01 00:00:00,1017.0,Electronics,3,44.68,West
1,2,2024-01-01 01:00:00,1154.0,Electronics,2,20.69,East
2,3,2024-01-01 02:00:00,1130.0,Apparel,4,41.60,West
3,4,2024-01-01 03:00:00,1087.0,Apparel,5,26.26,South
4,5,2024-01-01 04:00:00,1086.0,Apparel,2,39.45,West


In [3]:
orders.shape

(5015, 7)

## Step 2: Diagnosis first

Running the full diagnostic set before touching a single value.

In [4]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5015 entries, 0 to 3823
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          5015 non-null   int64         
 1   order_date        5015 non-null   datetime64[ns]
 2   customer_id       4865 non-null   float64       
 3   product_category  5015 non-null   object        
 4   quantity          5015 non-null   int64         
 5   unit_price        5015 non-null   float64       
 6   region            4817 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(2)
memory usage: 313.4+ KB


In [5]:
orders.describe()

,order_id,order_date,customer_id,quantity,unit_price
count,5015.000000,5015,4865.000000,5015.000000,5015.000000
mean,2500.953938,2024-04-14 03:57:14.177467648,1099.294347,3.937188,65.215825
min,1.000000,2024-01-01 00:00:00,1000.000000,-7.000000,-25.580000
25%,1250.500000,2024-02-22 01:30:00,1049.000000,2.000000,31.325000
50%,2502.000000,2024-04-14 05:00:00,1098.000000,4.000000,44.290000
75%,3751.500000,2024-06-05 06:30:00,1150.000000,6.000000,57.985000
max,5000.000000,2024-07-27 07:00:00,1199.000000,7.000000,4999.990000
std,1443.030494,NaN,57.795512,2.077920,320.661972


In [6]:
orders.isna().sum()

order_id              0
order_date            0
customer_id         150
product_category      0
quantity              0
unit_price            0
region              198
dtype: int64

In [7]:
orders['product_category'].value_counts()

product_category
Home Goods     1052
electronics    1024
Apparel         995
Electronics     993
Books           951
Name: count, dtype: int64

## Step 3: Clean it, column by column, issue by issue

Each problem found in diagnosis gets its own decision and its own justification — not one blanket rule applied to everything.

In [8]:
orders = orders.dropna(subset=['customer_id'])
orders.shape

(4865, 7)

**`customer_id` (150 missing values) → `dropna()`**

`customer_id` is an identifier, not a measurable quantity — there is no "typical" or "average" customer ID to fall back on the way a median works for a price. Filling it with a placeholder (like -1 or 0) would fabricate a fake customer that doesn't exist, which would corrupt any later `groupby('customer_id')` analysis. Since the missing rows are only ~3% of the dataset (150 of 5015), dropping them is a small, acceptable loss compared to inventing false identities.

In [9]:
orders['region'] = orders['region'].fillna('Unknown')
orders['region'].value_counts()

region
West       1210
East       1166
North      1163
South      1133
Unknown     193
Name: count, dtype: int64

**`region` (198 missing values) → `fillna('Unknown')`**

`region` is a categorical column, so a numeric fill like a median doesn't apply — the equivalent decision would be filling with the mode (the most frequent region). But the four regions are roughly evenly distributed (~24% each), so no single region is a strongly justified guess for what a missing row's true region was. Filling with the mode would artificially inflate that region's count and could distort any later region-based comparison. Filling with an explicit placeholder, `'Unknown'`, keeps the row (and its `amount`/`quantity` data usable) without fabricating a specific region that wasn't actually recorded.

In [10]:
orders['product_category'] = orders['product_category'].str.lower().str.title()
orders['product_category'].value_counts()

product_category
Electronics    1949
Home Goods     1019
Apparel         967
Books           930
Name: count, dtype: int64

**`product_category` casing inconsistency → vectorized `.str.lower().str.title()`**

`"Electronics"` and `"electronics"` were being treated as two separate categories purely due to casing, discovered via `.value_counts()`. Fixed with a single vectorized string operation across the whole column — `.str.lower()` normalizes all values to lowercase, then `.str.title()` capitalizes the first letter of every word (not just the first letter of the whole string, which `.str.capitalize()` would have done — this matters for multi-word categories like "Home Goods"). No manual row edits were needed.

In [11]:
orders['quantity'] = orders['quantity'].abs()
orders['quantity'].describe()

count    4865.000000
mean        3.983145
std         1.998386
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max         7.000000
Name: quantity, dtype: float64

**`quantity` negative values → `.abs()`**

30 rows had their quantity sign flipped to negative (a disguised-returns issue, not missing or random bad data) — the underlying magnitude was still correct, only the sign was wrong. Since there's no ambiguity about what the true value should have been, this isn't a drop/fill decision like a missing value — applying `.abs()` across the whole column corrects the sign with certainty rather than guessing a replacement value. Confirmed by `.describe()`: minimum quantity is now 1.0, with no negative values remaining.

In [12]:
orders.loc[orders['unit_price'] < 0, 'unit_price'] = np.nan
orders.loc[orders['unit_price'] > 500, 'unit_price'] = np.nan

orders['unit_price'] = orders['unit_price'].fillna(orders['unit_price'].median())
orders['unit_price'].describe()

count    4865.000000
mean       45.202164
std        19.147950
min         0.130000
25%        32.410000
50%        44.690000
75%        57.730000
max       128.020000
Name: unit_price, dtype: float64

**`unit_price` negative values and extreme outliers → treated as missing, then `fillna()` with median**

Two separate impossible-value problems existed in `unit_price`: negative prices (a price cannot be negative) and extreme outliers (4999.99, far outside the realistic 31–58 range), both data-quality errors with no known "true" underlying value — unlike `quantity`, where the sign flip was a known, reversible error. Since we can't reconstruct what the real price should have been, both problems were treated as missing data: negative values and values above a 500 threshold were set to `NaN`, then filled with the column's **median** rather than the mean. The mean was rejected specifically because it was itself still contaminated by the outliers at the time of calculation (65.21, pulled upward by the 4999.99 values) —  since the mean would have partly reproduced the very problem being fixed. After the fix, unit_price ranges from 0.13 to 128.02 with a mean of 45.20, consistent with the realistic 25th–75th percentile range seen before cleaning.

In [13]:
orders.isna().sum()

order_id            0
order_date          0
customer_id         0
product_category    0
quantity            0
unit_price          0
region              0
dtype: int64